# 04 - Kampanye tuning hyperparameter

Menjalankan grid yang dirancang di `tuning_grids/`. Prinsipnya human-in-the-loop:
tidak ada pencarian otomatis, urutan konfigurasi ditentukan manusia, dan setiap
baris hasil membawa kolom `catatan` yang merekam alasan konfigurasi itu dicoba.

Seleksi memakai split validation. Split test tidak disentuh sama sekali di
notebook ini.

Metode: grid kombinatorial untuk sumbu yang saling terkait, coordinate descent
untuk sumbu yang independen. Untuk RM-a, `lr`, `epochs`, dan `batch` bersama-sama
menentukan lintasan optimasi (batch 32 pada 5 epoch memberi separuh jumlah
langkah pembaruan dibanding batch 16), sehingga ketiganya harus digrid bersama;
`warmup_ratio` dan `weight_decay` efektif independen sehingga cukup dicoba satu
per satu di sel pemenang.

Prasyarat: `02_preprocessing.ipynb` sudah dijalankan.

In [ ]:
import pandas as pd

from src.config import settings
from src.services.campaign import CampaignRunner

OUT_DIR = settings.output_dir / "tuning_lite"

runner = CampaignRunner(
    out_dir=OUT_DIR, model_name="indobenchmark/indobert-lite-base-p2"
)
runner.write_hardware()
print("device   :", runner.device)
print("keluaran :", runner.out_dir)

## 1. Kalibrasi biaya

Jalankan satu konfigurasi RM-a lebih dulu untuk mengukur waktu dan memori
sesungguhnya di mesin ini, sebelum mempertaruhkan berjam-jam pada grid penuh.
Kalau memori kurang, turunkan `MICRO_BATCH` di `.env`; batch efektif tidak
berubah karena selisihnya ditutup akumulasi gradien.

In [ ]:
kalibrasi = runner.run(
    "rma",
    {"lr": 2e-5, "epochs": 5, "batch": 16, "warmup_ratio": 0.1, "weight_decay": 0.01},
    note="kalibrasi biaya: baseline kanonik, sekaligus run #1 grid",
)

per_run = kalibrasi["train_time_s"]
print(f"satu run RM-a: {per_run:.0f} s | peak {kalibrasi['peak_mem_mb']:.0f} MB")
print(f"perkiraan 26 run RM-a: {per_run * 26 / 60:.0f} menit")

## 2. Muat rancangan grid

In [ ]:
GRID_DIR = settings.data_dir.parent / "tuning_grids"

def muat_grid(nama: str) -> list[dict]:
    frame = pd.read_csv(GRID_DIR / nama)
    catatan = frame.pop("catatan") if "catatan" in frame.columns else ""
    return [
        {"config": {k: v for k, v in baris.items() if pd.notna(v)},
         "note": catatan.iloc[i] if hasattr(catatan, "iloc") else ""}
        for i, baris in enumerate(frame.to_dict("records"))
    ]

for berkas in sorted(GRID_DIR.glob("*.csv")):
    print(f"  {berkas.name}: {len(pd.read_csv(berkas))} konfigurasi")

## Melanjutkan kampanye yang terputus

`run_batch` menyaring konfigurasi yang sudah ada di riwayat secara default
(`resume=True`), sehingga sel batch di bawah aman dijalankan ulang apa adanya
setelah kernel mati, listrik padam, atau proses dihentikan. Yang sudah selesai
dilewati, penomoran run berlanjut, dan `best.json` tetap terjaga.

Yang hilang saat terputus hanyalah run yang sedang berjalan saat itu; run yang
sudah selesai ditulis atomik ke `runs_{skenario}.csv` begitu selesai.

Perbandingan memakai konfigurasi LENGKAP setelah nilai default diisi, dan untuk
RM-a `micro_batch` dinormalkan ke nilai efektifnya (`min(batch, micro_batch)`) —
nilai itulah yang menentukan ukuran batch di `DataLoader`.

Sel di bawah memperlihatkan apa yang tersisa sebelum batch dijalankan.

In [ ]:
def sisa(skenario: str, berkas: str) -> None:
    permintaan = muat_grid(berkas)
    tersisa = runner.pending_requests(skenario, permintaan)
    print(f"{berkas:34s} {len(permintaan) - len(tersisa):>3d}/{len(permintaan)} selesai, "
          f"{len(tersisa)} tersisa")

sisa("rma", "RMA_TUNING_GRID.csv")
sisa("rma", "RMA_TUNING_GRID_STAGE2.csv")
for berkas in ("RMB_TUNING_GRID.csv", "RMB_TUNING_GRID_STAGE1B.csv",
               "RMB_TUNING_GRID_STAGE2.csv", "RMB_TUNING_GRID_STAGE3.csv"):
    sisa("rmb", berkas)
for berkas in ("RMC_TUNING_GRID.csv", "RMC_TUNING_GRID_STAGE2.csv"):
    sisa("rmc", berkas)

## 3. RM-a

Grid tahap 1 menggarap tiga sumbu yang saling terkait. Konfigurasi yang gagal
diisolasi ke `runs_rma_errors.csv` dan tidak menghentikan sisa antrean.

In [ ]:
hasil_rma = runner.run_batch("rma", muat_grid("RMA_TUNING_GRID.csv"),
                             batch_id="rma_tahap1_grid")
hasil_rma.nlargest(10, "val_f1_macro")[
    ["run_id", "lr", "epochs", "batch", "val_f1_macro", "val_f1_judi",
     "train_time_s", "is_tie_with_best"]
]

Baca grid sebagai permukaan, bukan daftar. Heatmap `lr x epochs` per nilai
`batch` di `outputs/tuning/figures/` memperlihatkan apakah learning rate optimal
ikut bergeser saat batch berubah. Pemenang yang duduk di tepi grid adalah sinyal
untuk melebarkan rentang, bukan untuk langsung dikunci.

Selisih di bawah 0,15 pp dihitung seri karena hanya ada satu seed; pada kondisi
seri, pilih konfigurasi yang lebih murah.

In [ ]:
hasil_rma_tahap2 = runner.run_batch("rma", muat_grid("RMA_TUNING_GRID_STAGE2.csv"),
                                    batch_id="rma_tahap2_coordinate")
hasil_rma_tahap2[["run_id", "warmup_ratio", "weight_decay", "val_f1_macro",
                  "delta_vs_best_f1_macro_pp", "is_tie_with_best"]]

## 4. RM-b

In [ ]:
hasil_rmb = []
for berkas in ("RMB_TUNING_GRID.csv", "RMB_TUNING_GRID_STAGE1B.csv",
               "RMB_TUNING_GRID_STAGE2.csv", "RMB_TUNING_GRID_STAGE3.csv"):
    hasil_rmb.append(runner.run_batch("rmb", muat_grid(berkas),
                                      batch_id=berkas.replace(".csv", "").lower()))

pd.concat(hasil_rmb, ignore_index=True).nlargest(10, "val_f1_macro")[
    ["run_id", "head_arch", "hidden_dim", "lr", "epochs", "dropout",
     "val_f1_macro", "train_time_s", "trainable_params"]
]

## 5. RM-c

RM-c mewarisi head RM-b terbaik, jadi kampanye ini harus dijalankan SETELAH
RM-b selesai. Karena tidak ada training sama sekali, ratusan kombinasi
`alpha x k` selesai dalam hitungan detik.

In [ ]:
hasil_rmc = []
for berkas in ("RMC_TUNING_GRID.csv", "RMC_TUNING_GRID_STAGE2.csv"):
    hasil_rmc.append(runner.run_batch("rmc", muat_grid(berkas),
                                      batch_id=berkas.replace(".csv", "").lower()))

pd.concat(hasil_rmc, ignore_index=True).nlargest(10, "val_f1_macro")[
    ["run_id", "alpha", "k", "weighting", "val_f1_macro", "val_f1_judi", "eval_time_s"]
]

## 6. Juara tiap skenario

In [ ]:
import json

best = json.loads((OUT_DIR / "best.json").read_text(encoding="utf-8"))
for skenario, entri in best.items():
    print(f"{skenario}: run #{entri['run_id']} | val F1-macro {entri['val_f1_macro']:.4f}")
    print(f"     {entri['config']}\n")

In [ ]:
summary = json.loads((OUT_DIR / "tuning_summary.json").read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2, ensure_ascii=False))

## Ringkasan

Seluruh angka di atas berasal dari split validation. Split test masih tertutup
dan baru dibuka satu kali di `05_final_benchmark.ipynb`.

Kalau ingin menambah konfigurasi setelah membaca hasil, panggil `runner.run`
atau `runner.run_batch` lagi: riwayat menumpuk dan penomoran run berlanjut.